In [ ]:
# Import path configuration
# This provides all directory paths with automatic cluster/local detection
import sys
sys.path.insert(0, '.')
from python_config_template import (
    SPATIAL_BASE_DIR, SHARED_DIR, PLOTS_DIR,
    RAW_DATA_DIR, H5AD_OUTPUT_DIR, PAPER_OUTPUT_DIR,
    PAPER_H5AD_DIR, FINAL_OBJ_DIR, CELL_TYPES_DIR,
    TOPICS_DATA_PATH, SPATIAL_PLOTS_DIR,
    ENRICHMENT_BASE_DIR, ENRICHMENT_PROGRAMS_DIR, ENRICHMENT_PATHWAYS_DIR,
    ENRICHMENT_PROGRAMS_PATH, ENRICHMENT_PATHWAYS_PATH,
    CASE_SAMPLES, CONTROL_SAMPLES, SAMPLE_METADATA,
    ALL_ANNOTATED_PATH,
    get_gef_path
)


In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
import anndata as ad
from sklearn.mixture import GaussianMixture
import re
from sklearn.neighbors import NearestNeighbors
import glob


In [ ]:
print(f"Using paths from configuration:")
print(f"  H5AD output: {H5AD_OUTPUT_DIR}")
print(f"  Paper H5AD: {PAPER_H5AD_DIR}")

In [ ]:
# Load the annotated merged object using configured path
merge = sc.read_h5ad(ALL_ANNOTATED_PATH)

In [ ]:
AD_samples=merge[merge.obs['diagnosis']=='case'].obs['sample_id'].unique()
levels_df=merge.obs[['sample_id','levels']].drop_duplicates()

In [ ]:
#proccess each cell type- rescale 
import anndata as ad
adata_subtypes = {}

for cell_type in merge.obs["annotation"].unique():
    mask = merge.obs["annotation"] == cell_type
    adata_sub = merge[mask, :].copy()

    if merge.raw is not None:
        # Create an AnnData object from the raw layer
        adata_sub.raw = ad.AnnData(
            X = merge.raw.X[mask, :].copy(),
            var = merge.raw.var.copy(),
            obs = merge.obs[mask].copy()
        )

    adata_subtypes[cell_type] = adata_sub


In [ ]:
# For each cell type, process the data starting from the raw (normalized/log-transformed) matrix,
# so that each cell type is processed independently (variable genes, scaling, etc).

adata_subtypes_processed = {}

for cell_type, adata_sub in adata_subtypes.items():
    print(f"Processing {cell_type}...")

    # Prepare AnnData using raw values
    if adata_sub.raw is not None:
        # Use raw.X as the base matrix, with corresponding obs and var
        adata = ad.AnnData(
            X=adata_sub.raw.X.copy(),
            obs=adata_sub.obs.copy(),
            var=adata_sub.raw.var.copy()
        )



    # Find variable genes
    sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

    # Store the full data in .raw
    adata.raw = adata.copy()

    # Filter to variable genes
    adata = adata[:, adata.var.highly_variable].copy()

    # Regress out lenient covariates if present
    to_regress = [col for col in ['total_counts', 'pct_counts_mt'] if col in adata.obs.columns]
    if to_regress:
        sc.pp.regress_out(adata, to_regress)

    # Scale
    sc.pp.scale(adata, max_value=10, zero_center=True)

    # Store processed AnnData
    adata_subtypes_processed[cell_type] = adata
    


import os

save_dir = "ST_stereo/for_paper"
os.makedirs(save_dir, exist_ok=True)

for cell_type, adata in adata_subtypes_processed.items():
    file_path = f"{save_dir}/{cell_type}.h5ad"
    adata.write(file_path)
    print(f"Saved {cell_type} → {file_path}")




In [ ]:
adata_subtypes_processed = {}

for file_name in os.listdir(CELL_TYPES_DIR):
    if file_name.endswith(".h5ad"):
        cell_type = file_name.replace(".h5ad", "")
        adata_subtypes_processed[cell_type] = sc.read_h5ad(os.path.join(CELL_TYPES_DIR, file_name))



In [ ]:
#adding topics scores:

rel_topics_mic=["k13","k11","k8","k9"]
rel_topics_ast=["k5","k6"]
rel_topics_inh=['k7','k8','k10','k1']
rel_topics_excP=['k9','k10','k11','k4']
rel_topics_excM=['k4','k8','k3','k9']
rel_topic_oli=['k5','k6','k8']

df_mic = pd.read_hdf(TOPICS_DATA_PATH, key="/glia/microglia/de") 
df_mic=df_mic[df_mic["topic"].isin(rel_topics_mic)]
# Remove genes that are not unique within each dataframe's topic set
df_ast=pd.read_hdf(TOPICS_DATA_PATH, key="/glia/astrocytes/de") 
df_ast=df_ast[df_ast["topic"].isin(rel_topics_ast)]

df_inh=pd.read_hdf(TOPICS_DATA_PATH, key="/neuronal/inhibitory/de")
df_inh=df_inh[df_inh["topic"].isin(rel_topics_inh)]

df_oli=pd.read_hdf(TOPICS_DATA_PATH, key="/glia/oligodendrocytes/de")
df_oli=df_oli[df_oli["topic"].isin(rel_topic_oli)]

df_excM=pd.read_hdf(TOPICS_DATA_PATH, key="/neuronal/excitatory/cux2minus/de")
df_excM=df_excM[df_excM["topic"].isin(rel_topics_excM)]

df_excP=pd.read_hdf(TOPICS_DATA_PATH, key="/neuronal/excitatory/cux2plus/de")
df_excP=df_excP[df_excP["topic"].isin(rel_topics_excP)]

# Filter df_excM to keep only genes shared with excP, matching topics by position
# For each pair, find intersecting genes and keep only those in df_excM
shared_genes_per_topic_pair = {}
for i, (topic_excP, topic_excM) in enumerate(zip(rel_topics_excP, rel_topics_excM)):
    genes_excP = set(df_excP[df_excP["topic"] == topic_excP]["gene"].unique())
    genes_excM = set(df_excM[df_excM["topic"] == topic_excM]["gene"].unique())
    shared_genes = genes_excP.intersection(genes_excM)
    shared_genes_per_topic_pair[topic_excM] = shared_genes
    print(f"Topic pair {i}: {topic_excP} (excP) <-> {topic_excM} (excM): {len(shared_genes)} shared genes")

# Filter df_excM to keep only rows where gene is in the shared genes for its topic
df_excM = df_excM[df_excM.apply(lambda row: row["gene"] in shared_genes_per_topic_pair[row["topic"]], axis=1)]

def filter_unique_genes(df):
    return df[df.duplicated(subset=["gene"], keep=False) == False]

df_mic = filter_unique_genes(df_mic)
df_ast = filter_unique_genes(df_ast)
df_inh = filter_unique_genes(df_inh)
df_excM = filter_unique_genes(df_excM)
df_excP = filter_unique_genes(df_excP)
df_oli=filter_unique_genes(df_oli)

df_ast["cell_type"] = "AST"
df_mic["cell_type"] = "Microglia"
df_inh["cell_type"] = "INH"
df_excM["cell_type"]="Ex"
df_oli['cell_type']="OLIGO"



df_topics = pd.concat([df_ast, df_inh, df_mic,df_excM,df_oli], ignore_index=True)

In [ ]:
threshold_fraction = 0.02  # 2% of cells

# Compute expressed genes per cell type using raw counts by variavle genes
genes_per_celltype = {}
for ct in merge.obs["annotation"].unique():
    merge_ct = merge[merge.obs["annotation"] == ct]
    X = merge_ct.raw.X
    if hasattr(X, "toarray"):
        X = X.toarray()  # convert sparse to dense if needed

    min_cells = merge_ct.n_obs * threshold_fraction
    expressed_mask = (X > 0).sum(axis=0) > min_cells
    expressed_genes = merge_ct.raw.var_names[expressed_mask]
    genes_per_celltype[ct] =  adata_subtypes_processed[ct].var_names.intersection(expressed_genes)

# Compute topic scores within each cell type using raw counts
for ct in merge.obs["annotation"].unique():
    df_ct = df_topics[df_topics["cell_type"] == ct]
    if df_ct.empty:
        continue

    merge_sub = merge[merge.obs["annotation"] == ct].copy()

    for topic in df_ct["topic"].unique():
        topic_genes = df_ct.loc[df_ct["topic"] == topic, "gene"]
        topic_genes = [g for g in topic_genes if g in genes_per_celltype[ct]]

        if len(topic_genes) < 2:
            continue  # skip very small sets

        score_name = f"{ct}_{topic}_score"
        print(len(topic_genes))
        sc.tl.score_genes(merge_sub, gene_list=topic_genes, score_name=score_name, use_raw=True)

        # Store the scores back in the full AnnData
        merge.obs.loc[merge_sub.obs_names, score_name] = merge_sub.obs[score_name]

        print(f"Added score: {score_name} ({len(topic_genes)} genes)")




In [ ]:
score_cols = [c for c in merge.obs.columns if c.endswith("_score")]
thresholds = [0.75, 0.8, 0.85, 0.9]

for ct in merge.obs["annotation"].unique():
    cells_ct = merge.obs["annotation"] == ct

    for score in score_cols:
        vals = merge.obs.loc[cells_ct, score]

        for q in thresholds:
            col_name = f"{score}_bin{int(q*100)}"

            if vals.isna().all():
                merge.obs.loc[cells_ct, col_name] = np.nan
            else:
                thresh_val = np.nanquantile(vals, q)
                merge.obs.loc[cells_ct, col_name] = (vals >= thresh_val).astype(int)


In [ ]:
# adding signatures
signatures = {
    "Metallothioneins bind metals": ["MT3", "MT2A", "MT1E", "MT1X", "MT1G", "MT1M", "MT1F"],
    "cell cycle": ["CLASP2", "CENPC", "SEM1", "TPR", "STAG2", "NSL1", "AHCTF1", "ANAPC16", "SMC3", "ITGB3BP",
                   "BLZF1", "CEP290", "OFD1", "CEP70", "ANAPC4", "CEP63", "AKAP9", "LEMD3", "NUP205",
                   "PCM1", "WAPL", "CCP110", "DYNC1I2", "PSMD14", "RANBP2", "CEP57"],
    "DNA Repair": ["SEM1", "ATM", "POLK", "XPA", "SMARCA5", "BAZ1B", "INO80D", "COPS2", "WRN", "REV1", "PPP4R2", "ERCC5",
                   "RIF1", "RAD50", "HERC2", "XRCC5", "RFC1"],
    "MT genes": ['MT-TA', 'MT-TE', 'MT-ND6', 'MT-TQ', 'MT-ND5', 'MT-ND4L', 'MT-ND3'],
    "mitochondrial ribosome": ["MRPL51","MRPL57","MTG1","MRPS16","MRPS24","MRPL46","MRPS7","MRPS18C",
    "MRPL34","MRPL21" ,"MRPS11","MRPL54","MRPL33","MRPL19"],
    "mRNA transport":["DDX19A", "NPIPA1", "SMG6", "SARNP", "NUP98", "POM121C", "NUP214", "MVP", "POM121", 
    "SUPT6H", "ZC3H3", "NUP210", "NSUN2", "NUP133", "NUP85", "NUP188", "AKAP8L"],
    "postsynaptic specialization":["GRIN2D", "SHANK1", "IGSF9B", "DLGAP4", "MINK1", "CRTC1", "BCR", "TSC2", "SYNGAP1",
     "ADGRL1", "LRRC4B", "EEF2K", "CTNND1", "NEURL1", "PRR12", "CACNG7",
     "IQSEC3", "PRKCZ", "CACNA1C", "KALRN", "SYN3", "DTNB", "IQSEC1", "CABP1", "SHANK2", "CAMK2A", "PTPRS", "ATP2B2", "DLGAP2", "CPEB1"],
    "response to oxidative stress" : [
    "MT3", "APOE", "PRDX5", "HSPB1", "PTK2B", "PON2", "GPX4", "RPS3",
    "AGAP3", "MAPK3", "GPR37L1", "IPCEF1", "PRDX1", "TXNIP", "APOD",
    "PPIA", "CRYAB", "CAMKK2", "BNIP3", "GPX3", "RHOB", "HBB", "SELENOP",
    "HBA2", "PRDX2", "HBA1", "PPP2CB", "SOD2", "BAD", "PRNP"]
}


# Compute signature scores separately for INH and EX neurons

for ct in ["INH", "Ex"]:
    adata_ct = merge[merge.obs["annotation"] == ct].copy()
    for sig_name, genes in signatures.items():
        # Keep only genes that exist in your data
        genes_in_data = [g for g in genes if g in adata_subtypes_processed[ct].var_names]
        print(genes_in_data)
        if len(genes_in_data) == 0:
            print(f"No genes from '{sig_name}' found in {ct}")
            continue
        sc.tl.score_genes(adata_ct, gene_list=genes_in_data, score_name=f"{sig_name.replace(' ', '_')}_score", use_raw=True)
    
    # Add scores back to main AnnData
    for sig_name in signatures.keys():
        col = f"{sig_name.replace(' ', '_')}_score"
        if col in adata_ct.obs.columns:
            merge.obs.loc[adata_ct.obs_names, f"{ct}_{col}"] = adata_ct.obs[col]

In [ ]:

homeostatic_topics=['']
# Identify all columns matching the bimodal (no k#) {cell_type}_{pathway name}_score
pattern = re.compile(r'^([A-Za-z0-9]+)_(?!k\d+_)(.+)_score$')
bimodal_score_cols = []
col_cell_types = {}

for c in merge.obs.columns:
    m = pattern.match(c)
    if m:
        bimodal_score_cols.append(c)
        col_cell_types[c] = m.group(1)  # cell type
        
for score in bimodal_score_cols:
    cell_type = col_cell_types[score]
    cells_ct = merge.obs["annotation"] == cell_type
    vals = merge.obs.loc[cells_ct, score].dropna().values.reshape(-1,1)
    if len(vals) < 2:
        continue  # Can't fit GMM on less than 2 points


    try:
        gmm = GaussianMixture(n_components=2, random_state=0)
        gmm.fit(vals)
        means = gmm.means_.flatten()
        order = np.argsort(means)
        lower_component = order[0]
        higher_component = order[1]

        mean1, mean2 = means[lower_component], means[higher_component]

        # Create column for the peak value of higher distribution
        second_pick_col = f"{cell_type}_{score.split('_',1)[1]}_second_pick_bimodal_threshold"
        merge.obs.loc[cells_ct, second_pick_col] =(merge.obs.loc[cells_ct, score] >= mean2).astype(int) 
        merge.obs.loc[~cells_ct, second_pick_col] = np.nan

    except Exception as e:
        print(f"Error fitting GMM for {score}: {e}")



In [ ]:
merge

In [ ]:
merge.write(os.path.join(FINAL_OBJ_DIR, "spatial_obj_for_paper.h5ad"))

In [ ]:
merge=sc.read_h5ad(os.path.join(FINAL_OBJ_DIR, "spatial_obj_for_paper.h5ad"))

In [ ]:

def compute_cell_type_enrichment(
    adata,
    target_cell_type,
    neighbor_cell_type="Ex",
    ref_topic_cols=None,
    target_topic_cols=None,
    radius=200
):
    """
    Compute enrichment of neuronal topics around a target cell type (e.g., Microglia or Astrocytes).

    Parameters
    ----------
    adata : AnnData
        AnnData object with spatial coordinates in `obsm['spatial']` and topic scores in `.obs`.
    target_cell_type : str
        Name of the target cell type in `adata.obs['annotation']` to compute enrichment for.
    neighbor_cell_type : str
        Name of the neighbor cell type to consider (default 'Ex').
    microglia_topic_cols : list of str
        Columns corresponding to target cell type topics (e.g., microglia or astrocyte topics)
    neuron_topic_cols : list of str
        Columns corresponding to neuronal topics
    radius : float
        Radius (in spatial units) to define neighbors.
    
    Returns
    -------
    pd.DataFrame
        DataFrame with enrichment results for each target cell and topic pair.
    """
    microglia_topic_cols=ref_topic_cols
    neuron_topic_cols=target_topic_cols
    if microglia_topic_cols is None:
        microglia_topic_cols = [c for c in adata.obs.columns if c.startswith(target_cell_type[:3]) and c.endswith("_score_bin80")]
    if neuron_topic_cols is None:
        neuron_topic_cols = [c for c in adata.obs.columns if c.startswith(neighbor_cell_type) and c.endswith("_score_bin80")]
    
    results = []
    
    # Precompute neuronal topic densities per sample
    neuron_topic_density = {}
    for sample in adata.obs['sample_id'].unique():
        adata_sample = adata[adata.obs["sample_id"] == sample]
        neuron_mask = adata_sample.obs["annotation"] == neighbor_cell_type  # only neurons
        n_neurons = neuron_mask.sum()
        if n_neurons == 0:
            continue
        densities = {nt: (adata_sample.obs.loc[neuron_mask, nt] > 0).sum() / n_neurons for nt in neuron_topic_cols}
        neuron_topic_density[sample] = densities
        
    # Loop over samples
    for sample in adata.obs['sample_id'].unique():
        adata_sample = adata[adata.obs["sample_id"] == sample].copy()
        coords = adata_sample.obsm.get("spatial")
        if coords is None:
            continue

        # Target cells
        target_mask = adata_sample.obs["annotation"] == target_cell_type
        if target_mask.sum() == 0:
            continue

        coords_target = coords[target_mask]
        target_indices = np.where(target_mask)[0]

        # Build neighborhood structure
        nbrs = NearestNeighbors(radius=radius, algorithm="ball_tree").fit(coords)
        neighbor_indices = nbrs.radius_neighbors(coords_target, return_distance=False)

        annotation_all = adata_sample.obs["annotation"].values
        target_indices_filtered = []
        neighbor_indices_filtered = []

        for t_idx, neigh_idx in zip(target_indices, neighbor_indices):
            neigh_idx = neigh_idx[neigh_idx != t_idx]
            if len(neigh_idx) == 0:
                continue
            if np.any(annotation_all[neigh_idx] == neighbor_cell_type):
                target_indices_filtered.append(t_idx)
                neighbor_indices_filtered.append(neigh_idx)

        if len(target_indices_filtered) == 0:
            continue

        # Compute observed & expected counts
        for t_idx, neigh_idx in zip(target_indices_filtered, neighbor_indices_filtered):
            cell_id = adata_sample.obs.index[t_idx]
            active_topics = [t for t in microglia_topic_cols if adata_sample.obs.loc[cell_id, t] > 0]
            if not active_topics:
                continue

            neighbor_obs = adata_sample.obs.iloc[neigh_idx]
        
            neighbor_neurons = neighbor_obs[neighbor_obs['annotation'] == neighbor_cell_type]
            n_neighbors = neighbor_neurons.shape[0]

            for target_topic in active_topics:
                for neuron_topic in neuron_topic_cols:
                    observed = int((neighbor_obs[neuron_topic] > 0).sum())
                    expected = neuron_topic_density[sample][neuron_topic] * n_neighbors
                    enrichment = observed / expected if expected > 0 else np.nan

                    results.append({
                        "sample_id": sample,
                        f"{target_cell_type}_cell": cell_id,
                        f"{target_cell_type}_topic": target_topic,
                        "neuron_topic": neuron_topic,
                        "n_neighbors": n_neighbors,
                        "observed": observed,
                        "expected": expected,
                        "enrichment": enrichment
                    })
    
    return pd.DataFrame(results)


In [ ]:
from scipy.stats import shapiro, ttest_1samp
from statsmodels.stats.multitest import multipletests


def compute_celltype_enrichment_per_sample(results_df, levels_df, pdf_path, cell_type):
    """
    Compute enrichment statistics (mean per sample, Shapiro, t-test vs 1, BH) for
    each neuron_topic without generating plots.

    Parameters
    ----------
    results_df : pd.DataFrame
        Enrichment results with columns including `sample_id`, `neuron_topic`,
        and `{cell_type}_topic`.
    levels_df : pd.DataFrame
        Sample-level metadata (kept for signature compatibility; not used here).
    pdf_path : str
        Unused placeholder to keep the function signature stable with callers.
    cell_type : str
        The target cell type (used to find the relevant topic column).

    Returns
    -------
    pd.DataFrame
        One row per neuron_topic within each ref_topic, containing mean
        enrichment across samples, raw and BH-corrected p-values, Shapiro stats,
        and per-sample mean enrichment columns.
    """

    cell_topic_col = f"{cell_type}_topic"
    topics = results_df[cell_topic_col].unique()

    all_pvals = []

    for topic in topics:
        df_sub = results_df[results_df[cell_topic_col] == topic].copy()
        if df_sub.empty:
            continue

        # Mean enrichment per (sample, neuron_topic)
        summary = (
            df_sub.groupby(["sample_id", "neuron_topic"])["enrichment"]
            .mean()
            .reset_index()
        )

        # Wide matrix of per-sample means to attach later
        sample_means_wide = summary.pivot(
            index="neuron_topic", columns="sample_id", values="enrichment"
        )

        neuron_topics = summary["neuron_topic"].unique()
        p_rows = []

        for nt in neuron_topics:
            vals = summary.loc[summary["neuron_topic"] == nt, "enrichment"].dropna()
            if len(vals) == 0:
                continue

            # Normality (only meaningful for n>=3)
            try:
                W, p_norm = shapiro(vals) if len(vals) >= 3 else (np.nan, np.nan)
            except Exception:
                W, p_norm = (np.nan, np.nan)

            try:
                _, p = ttest_1samp(vals, popmean=1)
            except ValueError:
                p = 1.0

            p_rows.append(
                {
                    "ref_topic": topic,
                    "neuron_topic": nt,
                    "mean_enrichment": vals.mean(),
                    "p_raw": p,
                    "shapiro_W": W,
                    "shapiro_p": p_norm,
                }
            )

        if not p_rows:
            continue

        p_df = pd.DataFrame(p_rows)

        # BH correction within this ref_topic
        _, p_bh, _, _ = multipletests(p_df["p_raw"], method="fdr_bh")
        p_df["p_bh"] = p_bh

        # Add per-sample mean enrichment columns
        p_df = p_df.merge(
            sample_means_wide, left_on="neuron_topic", right_index=True, how="left"
        )

        all_pvals.append(p_df)

    if not all_pvals:
        return pd.DataFrame()

    out = pd.concat(all_pvals, ignore_index=True)
    out = out.rename(columns={"neuron_topic": "neighbor_topic"})

    return out


In [ ]:

out_dir = "ST_stereo/for_paper/enrichment_data_programs"
os.makedirs(out_dir, exist_ok=True)

for bin in ["_score_bin75","_score_bin80","_score_bin85","_score_bin90"]:

    ex_topic_cols=[c for c in merge.obs.columns if c.startswith("Ex") and c.endswith(bin)]
    inh_topic_cols=[c for c in merge.obs.columns if c.startswith("INH") and c.endswith(bin)]
    microglia_topic_cols = [c for c in merge.obs.columns if c.startswith("Microglia") and c.endswith(bin)]
    ast_topic_cols = [c for c in merge.obs.columns if c.startswith("AST") and c.endswith(bin)]


    neuron_neuron_results = compute_cell_type_enrichment(
        merge, target_cell_type="Ex", neighbor_cell_type="Ex",ref_topic_cols=ex_topic_cols, target_topic_cols=ex_topic_cols
    )
    ex_microglia_results = compute_cell_type_enrichment(
        merge, target_cell_type="Ex", neighbor_cell_type="Microglia",ref_topic_cols=ex_topic_cols, target_topic_cols=microglia_topic_cols
    )
    microglia_ex_results = compute_cell_type_enrichment(
        merge, target_cell_type="Microglia", neighbor_cell_type="Ex", ref_topic_cols=microglia_topic_cols, target_topic_cols=ex_topic_cols
    )
    ex_ast_results = compute_cell_type_enrichment(
        merge, target_cell_type="Ex", neighbor_cell_type="AST",ref_topic_cols=ex_topic_cols, target_topic_cols=ast_topic_cols
    )
    ast_ex_results = compute_cell_type_enrichment(
        merge, target_cell_type="AST", neighbor_cell_type="Ex", ref_topic_cols=ast_topic_cols, target_topic_cols=ex_topic_cols
    )
    ast_mic_results = compute_cell_type_enrichment(
        merge, target_cell_type="AST", neighbor_cell_type="Microglia",ref_topic_cols=ast_topic_cols, target_topic_cols=microglia_topic_cols
    )
    mic_mic_results = compute_cell_type_enrichment(
        merge, target_cell_type="Microglia", neighbor_cell_type="Microglia", ref_topic_cols=microglia_topic_cols, target_topic_cols=microglia_topic_cols
    )
    mic_ast_results = compute_cell_type_enrichment(
            merge, target_cell_type="Microglia", neighbor_cell_type="AST",ref_topic_cols=microglia_topic_cols, target_topic_cols=ast_topic_cols
        )
    neuron_neuron_INH_results = compute_cell_type_enrichment(
        merge, target_cell_type="INH", neighbor_cell_type="INH",ref_topic_cols=inh_topic_cols, target_topic_cols=inh_topic_cols
    )

    inh_microglia_results = compute_cell_type_enrichment(
        merge, target_cell_type="INH", neighbor_cell_type="Microglia",ref_topic_cols=inh_topic_cols, target_topic_cols=microglia_topic_cols
    )

    microglia_inh_results = compute_cell_type_enrichment(
        merge, target_cell_type="Microglia", neighbor_cell_type="INH", ref_topic_cols=microglia_topic_cols, target_topic_cols=inh_topic_cols
    )

    inh_ast_results = compute_cell_type_enrichment(
        merge, target_cell_type="INH", neighbor_cell_type="AST",ref_topic_cols=inh_topic_cols, target_topic_cols=ast_topic_cols
    )

    ast_inh_results = compute_cell_type_enrichment(
        merge, target_cell_type="AST", neighbor_cell_type="INH", ref_topic_cols=ast_topic_cols, target_topic_cols=inh_topic_cols
    )

    inh_ex_results = compute_cell_type_enrichment(
        merge, target_cell_type="INH", neighbor_cell_type="Ex", ref_topic_cols=inh_topic_cols, target_topic_cols=ex_topic_cols
    )
    ex_inh_results = compute_cell_type_enrichment(
        merge, target_cell_type="Ex", neighbor_cell_type="INH", ref_topic_cols=ex_topic_cols, target_topic_cols=inh_topic_cols
    )
 

    # ------- SAVE RESULTS AS CSV --------
    results_dict = {
        "Ex_Ex": neuron_neuron_results,
        "Ex_Microglia": ex_microglia_results,
        "Microglia_Ex": microglia_ex_results,
        "Ex_AST": ex_ast_results,
        "AST_Ex": ast_ex_results,
        "AST_Microglia": ast_mic_results,
        "Microglia_Microglia": mic_mic_results,
        "Microglia_AST":mic_ast_results,
        "INH_INH": neuron_neuron_INH_results,
        "INH_Microglia": inh_microglia_results,
        "Microglia_INH": microglia_inh_results,
        "INH_AST": inh_ast_results,
        "AST_INH": ast_inh_results,
        "INH_Ex": inh_ex_results,
        "Ex_INH": ex_inh_results,

    }

    for name, df in results_dict.items():
        save_path = f"{out_dir}/{name}_enrichment_{bin}.csv"
        df.to_csv(save_path, index=False)





In [ ]:
bin="_score_bin80"
ex_topic_cols=[c for c in merge.obs.columns if c.startswith("Ex") and c.endswith(bin)]
inh_topic_cols=[c for c in merge.obs.columns if c.startswith("INH") and c.endswith(bin)]
microglia_topic_cols = [c for c in merge.obs.columns if c.startswith("Microglia") and c.endswith(bin)]
ast_topic_cols = [c for c in merge.obs.columns if c.startswith("AST") and c.endswith(bin)]


ast_ex_results = compute_cell_type_enrichment(
    merge, target_cell_type="AST", neighbor_cell_type="Ex", ref_topic_cols=ast_topic_cols, target_topic_cols=ex_topic_cols
)

In [ ]:
ast_ex_results

In [ ]:
out_dir = ENRICHMENT_BASE_DIR
os.makedirs(out_dir, exist_ok=True)

bin="_score_bin80"
ex_topic_cols = [c for c in merge.obs.columns if c.startswith("Ex") and c.endswith('_second_pick_bimodal_threshold')]
inh_topic_cols=[c for c in merge.obs.columns if c.startswith("INH") and c.endswith('_second_pick_bimodal_threshold')]

microglia_topic_cols = [c for c in merge.obs.columns if c.startswith("Microglia") and c.endswith(bin)]
ast_topic_cols = [c for c in merge.obs.columns if c.startswith("AST") and c.endswith(bin)]


neuron_neuron_results = compute_cell_type_enrichment(
    merge, target_cell_type="Ex", neighbor_cell_type="Ex",ref_topic_cols=ex_topic_cols, target_topic_cols=ex_topic_cols
)
print("microglia")
ex_microglia_results = compute_cell_type_enrichment(
    merge, target_cell_type="Ex", neighbor_cell_type="Microglia",ref_topic_cols=ex_topic_cols, target_topic_cols=microglia_topic_cols
)

microglia_ex_results = compute_cell_type_enrichment(
    merge, target_cell_type="Microglia", neighbor_cell_type="Ex", ref_topic_cols=microglia_topic_cols, target_topic_cols=ex_topic_cols
)

ex_ast_results = compute_cell_type_enrichment(
    merge, target_cell_type="Ex", neighbor_cell_type="AST",ref_topic_cols=ex_topic_cols, target_topic_cols=ast_topic_cols
)

ast_ex_results = compute_cell_type_enrichment(
    merge, target_cell_type="AST", neighbor_cell_type="Ex", ref_topic_cols=ast_topic_cols, target_topic_cols=ex_topic_cols
)

mic_ast_results = compute_cell_type_enrichment(
        merge, target_cell_type="Microglia", neighbor_cell_type="AST",ref_topic_cols=microglia_topic_cols, target_topic_cols=ast_topic_cols
    )

neuron_neuron_INH_results = compute_cell_type_enrichment(
    merge, target_cell_type="INH", neighbor_cell_type="INH",ref_topic_cols=inh_topic_cols, target_topic_cols=inh_topic_cols
)

inh_microglia_results = compute_cell_type_enrichment(
    merge, target_cell_type="INH", neighbor_cell_type="Microglia",ref_topic_cols=inh_topic_cols, target_topic_cols=microglia_topic_cols
)

microglia_inh_results = compute_cell_type_enrichment(
    merge, target_cell_type="Microglia", neighbor_cell_type="INH", ref_topic_cols=microglia_topic_cols, target_topic_cols=inh_topic_cols
)

inh_ast_results = compute_cell_type_enrichment(
    merge, target_cell_type="INH", neighbor_cell_type="AST",ref_topic_cols=inh_topic_cols, target_topic_cols=ast_topic_cols
)

ast_inh_results = compute_cell_type_enrichment(
    merge, target_cell_type="AST", neighbor_cell_type="INH", ref_topic_cols=ast_topic_cols, target_topic_cols=inh_topic_cols
)

inh_ex_results = compute_cell_type_enrichment(
    merge, target_cell_type="INH", neighbor_cell_type="Ex", ref_topic_cols=inh_topic_cols, target_topic_cols=ex_topic_cols
)
ex_inh_results = compute_cell_type_enrichment(
    merge, target_cell_type="Ex", neighbor_cell_type="INH", ref_topic_cols=ex_topic_cols, target_topic_cols=inh_topic_cols
)


 # ------- SAVE RESULTS AS CSV --------
results_dict = {
    "Ex_Ex": neuron_neuron_results,
    "Ex_Microglia": ex_microglia_results,
    "Microglia_Ex": microglia_ex_results,
    "Ex_AST": ex_ast_results,
    "AST_Ex": ast_ex_results,
    "INH_INH": neuron_neuron_INH_results,
    "INH_Microglia": inh_microglia_results,
    "Microglia_INH": microglia_inh_results,
    "INH_AST": inh_ast_results,
    "AST_INH": ast_inh_results,
    "INH_Ex": inh_ex_results,
    "Ex_INH": ex_inh_results,

}

for name, df in results_dict.items():
    save_path = f"{out_dir}/{name}_enrichment_pathways_thresold.csv"
    df.to_csv(save_path, index=False)





In [ ]:
for name, df in results_dict.items():
    save_path = f"{out_dir}/{name}_enrichment_pathways_thresold.csv"
    df.to_csv(save_path, index=False)


In [ ]:
def load_enrichment_by_bin(out_dir):
    """
    Reads all files of form:
        {name}_enrichment_{bin}.csv
    and organizes them into:
        data[bin][name] = DataFrame
    """
    data = {}

    # Find all enrichment CSVs in the directory
    files = glob.glob(os.path.join(out_dir, "*_enrichment_*.csv"))
    

    for path in files:
        # Extract name + bin from filename
        filename = os.path.basename(path)
        name_part, bin_part = filename.replace(".csv", "").split("_enrichment_")
        
        name = name_part                     # e.g., "Ex_Ex"
        bin_id = bin_part                    # e.g., "3", "bin4", etc.

        # Initialize bin group if needed
        if bin_id not in data:
            data[bin_id] = {}

        # Read the file
        df = pd.read_csv(path)

        # Store inside nested dict
        data[bin_id][name] = df

    return data


In [ ]:
programs_data = load_enrichment_by_bin(ENRICHMENT_PROGRAMS_DIR)
pathways_data = load_enrichment_by_bin(ENRICHMENT_PATHWAYS_DIR)

In [ ]:
programs_data

In [ ]:
pdf_out_dir = ENRICHMENT_PROGRAMS_DIR
os.makedirs(pdf_out_dir, exist_ok=True)

all_pval_results = []

for bin_name, pairs_dict in programs_data.items():

    print(f"Processing bin: {bin_name}")

    for pair_name, df in pairs_dict.items():

        # Extract the first cell type from a key like: "Ex_Microglia"
        cell_type = pair_name.split("_")[0]

        # Build output PDF path
        pdf_path = os.path.join(pdf_out_dir, f"{pair_name}_{bin_name}.pdf")

        print(f"  → computing {pair_name} (cell_type={cell_type}) → {pdf_path}")

        # Run your function
        pvals_df=compute_celltype_enrichment_per_sample(
            results_df=df[df["sample_id"].isin(AD_samples)],
            levels_df=levels_df,
            pdf_path=pdf_path,
            cell_type=cell_type
        )

        pvals_df["bin"] = bin_name   # Add the bin as a column
        pvals_df["pair"] = pair_name # Optionally add the pair name

        all_pval_results.append(pvals_df)
    
all_pvals_df = pd.concat(all_pval_results, ignore_index=True)
#Save the DataFrame all_pvals_df to the output directory as a CSV for reference
all_pvals_df.to_csv(os.path.join(pdf_out_dir, "all_pvals_programs.csv"), index=False)


pdf_out_dir = ENRICHMENT_BASE_DIR
os.makedirs(pdf_out_dir, exist_ok=True)

all_pval_results = []

for bin_name, pairs_dict in pathways_data.items():

    print(f"Processing bin: {bin_name}")

    for pair_name, df in pairs_dict.items():

        # Extract the first cell type from a key like: "Ex_Microglia"
        cell_type = pair_name.split("_")[0]

        # Build output PDF path
        pdf_path = os.path.join(pdf_out_dir, f"{pair_name}_{bin_name}.pdf")

        print(f"  → computing {pair_name} (cell_type={cell_type}) → {pdf_path}")

        # Run your function
        pvals_df = compute_celltype_enrichment_per_sample(
            results_df=df[df["sample_id"].isin(AD_samples)],
            levels_df=levels_df,
            pdf_path=pdf_path,
            cell_type=cell_type
        )

        pvals_df["bin"] = bin_name   # Add the bin as a column
        pvals_df["pair"] = pair_name # Optionally add the pair name

        all_pval_results.append(pvals_df)
    
all_pvals_df = pd.concat(all_pval_results, ignore_index=True)
#Save the DataFrame all_pvals_df to the output directory as a CSV for reference
all_pvals_df.to_csv(os.path.join(pdf_out_dir, "all_pvals_pathways.csv"), index=False)